In [4]:
%pwd

'c:\\Users\\Amal\\Desktop\\End to End project Mlops mlflow github\\End-to-End-project-Mlops-Mlflow\\research'

Test du fichier prediction.py

In [5]:
%cd ..

c:\Users\Amal\Desktop\End to End project Mlops mlflow github\End-to-End-project-Mlops-Mlflow


C:\Users\Amal\AppData\Roaming\Python\Python39\site-packages\IPython\core\magics\osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [6]:
%pwd

'c:\\Users\\Amal\\Desktop\\End to End project Mlops mlflow github\\End-to-End-project-Mlops-Mlflow'

In [23]:
import joblib 
import pandas as pd
from pathlib import Path

In [25]:
df_transf_data = pd.read_csv(Path("artifacts/data_transformation/transf_data_file.csv"))
df_transf_data

,Unnamed: 0,mean_input_data,max_input_data,min_input_data
0,area,5150.541284,16200,1650
1,bedrooms,2.965138,6,1
2,bathrooms,1.286239,4,1
3,stories,1.805505,4,1
4,mainroad,0.858716,1,0
5,guestroom,0.177982,1,0
6,basement,0.350459,1,0
7,hotwaterheating,0.045872,1,0
8,airconditioning,0.315596,1,0
9,parking,0.693578,3,0


In [26]:
#  input_data = [area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus]

input_data = [7420,4,2,3,"yes","no","no","no","yes",2,"yes","furnished"]

In [27]:
column_name = ["mainroad", "guestroom", "basement", "hotwaterheating",
               "airconditioning", "prefarea", "furnishingstatus"]

In [28]:
encoder = joblib.load(Path("artifacts/data_transformation/encoder.pkl"))

In [29]:
input_data = [7420,4,2,3,"yes","no","no","no","yes",2,"yes","furnished"]

In [30]:
str_values = [x for x in input_data if isinstance(x, str)]

encoded_str_values = []

for value, col in zip(str_values, column_name):
    encoded_value = encoder[col].transform([value])[0]
    encoded_str_values.append(encoded_value)

print(encoded_str_values)

[1, 0, 0, 0, 1, 1, 0]


In [31]:
# remplaer str dans input data par [1, 0, 0, 0, 1, 1, 0]

print("Original:", input_data)

# Récupérer les indices des valeurs string
string_indices = [i for i, value in enumerate(input_data) if isinstance(value, str)]

# Remplacer chaque valeur string par la valeur correspondante du mapping
for idx, map_val in zip(string_indices, encoded_str_values):
    input_data[idx] = map_val

print("Encodé:  ", input_data)

Original: [7420, 4, 2, 3, 'yes', 'no', 'no', 'no', 'yes', 2, 'yes', 'furnished']
Encodé:   [7420, 4, 2, 3, 1, 0, 0, 0, 1, 2, 1, 0]


In [32]:
input_normalized_data = (input_data - df_transf_data ["mean_input_data"]) / (df_transf_data ["max_input_data"] - df_transf_data ["min_input_data"])

In [33]:
input_normalized_data

0     0.155977
1     0.206972
2     0.237920
3     0.398165
4     0.141284
5    -0.177982
6    -0.350459
7    -0.045872
8     0.684404
9     0.435474
10    0.765138
11   -0.534862
dtype: float64

In [36]:
import numpy as np

In [34]:
# predect new data
model = joblib.load(Path('artifacts/model_trainer/model.joblib'))

In [41]:
model.predict(np.array(input_normalized_data).reshape(1, 12))

array([6865222.0876264])

Define a class

In [7]:
import joblib 
import pandas as pd
from pathlib import Path
import numpy as np

In [8]:
column_name = ["mainroad", "guestroom", "basement", "hotwaterheating",
               "airconditioning", "prefarea", "furnishingstatus"]

In [9]:
# Ajouter méthode de noramlisation, PCA etc si vous avez en training
class PredictionPipeline:
    def __init__(self):
        self.model = joblib.load(Path('artifacts/model_trainer/model.joblib'))
        self.df_transf_data = pd.read_csv(Path("artifacts/data_transformation/transf_data_file.csv"))
        self.encoder =  joblib.load(Path("artifacts/data_transformation/encoder.pkl"))
        
    def Normalize_input_data(self, input_data):
        
        str_values = [x for x in input_data if isinstance(x, str)]

        encoded_str_values = []

        for value, col in zip(str_values, column_name):
            encoded_value = self.encoder[col].transform([value])[0]
            encoded_str_values.append(encoded_value)

        # remplaer str dans input data par [1, 0, 0, 0, 1, 1, 0]
        # Récupérer les indices des valeurs string
        string_indices = [i for i, value in enumerate(input_data) if isinstance(value, str)]

        # Remplacer chaque valeur string par la valeur correspondante du mapping
        for idx, map_val in zip(string_indices, encoded_str_values):
            input_data[idx] = map_val

        input_normalized_data = (input_data - self.df_transf_data ["mean_input_data"]) / (self.df_transf_data ["max_input_data"] - self.df_transf_data ["min_input_data"])
        
        return input_normalized_data
    
    def predict(self, input_normalized_data):
        prediction = self.model.predict(np.array(input_normalized_data).reshape(1, 12))

        return prediction

In [18]:
input_data_1 = [7420,4,2,3,"yes","no","no","no","yes",2,"yes","furnished"]

In [19]:
data_pred = PredictionPipeline()
    
data_normalized_1 = data_pred. Normalize_input_data(input_data_1)

data_pred.predict(data_normalized_1)

array([6865222.0876264])